## Test PyTorch

In [1]:
def print_gpu_info():
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        print("Current device:", torch.cuda.current_device())
        print("GPU name:", torch.cuda.get_device_name(0))

        x = torch.randn(1000, 1000, device="cuda")
        y = torch.randn(1000, 1000, device="cuda")
        z = x @ y

        torch.cuda.synchronize()
        print("Success! Tensor is on:", z.device)
    else:
        print("No GPU available.")


## Setup logging

In [2]:
import logging, os, sys

from dask.typing import Key
from distributed import Worker, WorkerPlugin


def configure_logging(config_file, verbose, logger_name=None):
    if config_file and os.path.exists(config_file):
        print(f'Configure logging using {config_file}, logger name: {logger_name}')
        logging.config.fileConfig(config_file)
    else:
        print(f'Configure logging using basic config - verbose: {verbose}, logger name: {logger_name}')
        log_format = '%(asctime)s - %(threadName)s:%(name)s - %(levelname)s - %(message)s'
        log_level = logging.DEBUG if verbose else logging.INFO
        logging.basicConfig(level=log_level,
                            format=log_format,
                            datefmt='%Y-%m-%d %H:%M:%S',
                            handlers=[
                                logging.StreamHandler(stream=sys.stdout)
                            ])
    return logging.getLogger(logger_name)


class ConfigureWorkerPlugin(WorkerPlugin):
    def __init__(self, models_dir, logging_config, verbose):
        self.models_dir = models_dir
        self.logging_config = logging_config
        self.verbose = verbose
        self.logger = None

    def setup(self, worker: Worker):
        self.logger = configure_logging(self.logging_config, self.verbose,
                                        logger_name='dask_worker')
        if self.models_dir:
            self.logger.info(f'Set cellpose models path: {self.models_dir}')
            os.environ['CELLPOSE_LOCAL_MODELS_PATH'] = self.models_dir

    def teardown(self, worker: Worker):
        pass

    def transition(self, key: Key, start: str, finish: str, **kwargs):
        pass

    def release_key(self, key: str, state: str, cause: str | None, reason: None, report: bool):
        pass


log_config_file = ''
verbose_logging = True

logger = configure_logging(log_config_file, verbose_logging)


Configure logging using basic config - verbose: True, logger name: None


In [3]:
import numpy as np
import numcodecs as zv2codecs
import zarr
import zarr.codecs as zv3codecs


def open_zarr(img_container_path, img_subpath, mode='r') -> zarr.Array|zarr.Group:
    img_container = zarr.open(store=img_container_path, mode=mode)
    if img_subpath:
        return img_container[img_subpath]
    else:
        return img_container


def create_zarr(zarr_container_path, dataset_subpath,
                dataset_3dshape,
                dtype='uint32',
                chunks=(64,64,64),
                sharding_factors=(4,4,4),
                zarr_format=2,
                mode='a'):

    logger.info(f'Create zarr container {zarr_container_path}:{dataset_subpath}')
    zarr_container = zarr.open_group(
        store=zarr_container_path,
        mode=mode,
        zarr_format=zarr_format,
    )

    if zarr_format == 2:
        zstd_codec = zv2codecs.get_codec({'id': 'zstd', 'level': 5})
        compressors = {'compressors': zstd_codec}
        chunk_key_separator = {'name': 'v2', 'separator': '/'}
        sharding_args = {}
    else:
        zstd_codec = zv3codecs.ZstdCodec(level=5)
        compressors = {'compressors': (zstd_codec,)}
        chunk_key_separator = None
        shard_shape = tuple(int(x) for x in np.asarray(chunks) * sharding_factors)
        sharding_args = {'shards': shard_shape}
        logger.info(f'Use zarrv3 sharding: {sharding_args}')

    return zarr_container.create_array(
        name=dataset_subpath,
        shape=dataset_3dshape,
        chunks=chunks,
        dtype=dtype,
        chunk_key_encoding=chunk_key_separator,
        **compressors,
        **sharding_args,
    )

2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'zlib'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'gzip'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'bz2'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'lzma'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'blosc'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'zstd'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'lz4'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'astype'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'delta'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'quantize'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'fixedscaleoffset'
2026-07-22 11:22:13 - MainThread:numcodecs - DEBUG - Registering codec 'packbits'
2026-07-22 11:22:13 - MainThread:numcodecs

## Method to create the dask cluster

In [4]:
from cellpose.contrib.distributed_segmentation import janeliaLSFCluster, myLocalCluster


def create_cluster(nworkers:int,
                   cluster_type:str='lsf',
                   config={},
                   worker_plugin=None,
                   ncpus:int=1,
                   **kwargs):
    if cluster_type == 'lsf':
        dask_cluster = janeliaLSFCluster(
            ncpus, # ncpus per worker
            1, # min workers
            nworkers,
            config=config,
            **kwargs,
        )
    else:
        dask_cluster = myLocalCluster(
            1,
            n_workers=nworkers,
            threads_per_worker=1,
            processes=True,
            host="localhost",
            **kwargs,
        )
    if worker_plugin is not None:
        dask_cluster.client.register_plugin(worker_plugin, 'WorkerConfig')
    
    return dask_cluster


2026-07-22 11:22:16 - MainThread:matplotlib - DEBUG - matplotlib data path: /groups/scicompsoft/home/goinac/miniforge3/envs/cellpose-tools/lib/python3.12/site-packages/matplotlib/mpl-data
2026-07-22 11:22:16 - MainThread:matplotlib - DEBUG - CONFIGDIR=/groups/scicompsoft/home/goinac/.config/matplotlib
2026-07-22 11:22:16 - MainThread:matplotlib - DEBUG - interactive is False
2026-07-22 11:22:16 - MainThread:matplotlib - DEBUG - platform is linux
2026-07-22 11:22:16 - MainThread:matplotlib - DEBUG - CACHEDIR=/groups/scicompsoft/home/goinac/.cache/matplotlib
2026-07-22 11:22:16 - MainThread:matplotlib.font_manager - DEBUG - Using fontManager instance from /groups/scicompsoft/home/goinac/.cache/matplotlib/fontlist-v3.11.0.json


## Invoke distributed segmentation

In [5]:

models_dir = ''
logging_config = ''
nworkers = 2
ncpus = 2

worker_plugin = ConfigureWorkerPlugin(models_dir, logging_config, True)

worker_directives = [
    "-P scicompsoft",
    "-gpu 'num=1'",
    "-q gpu_l4",
    "-J cellpose-worker",
]

lsf_worker_job_args = {
    'cluster_type': 'lsf',
    'job_extra_directives': worker_directives,
}

local_worker_job_args = {
    'cluster_type': 'local',
}

dask_config = {
    'distributed.worker.memory.target':0.9,
    'distributed.worker.memory.spill':0.9,
    'distributed.worker.memory.pause':0.9,
    'distributed.comm.retry.count':10,
    'distributed.comm.timeouts.connect':'600s',
    'distributed.scheduler.unknown-task-duration':'60m',
}

worker_job_args = {
    'worker_plugin': worker_plugin,
    'death_timeout':1200,
    'config': dask_config,
    **lsf_worker_job_args,
}


In [6]:
from segmentation.distributed_cellpose import distributed_eval


def run_distributed_eval(input_path, input_subpath, output_path, output_subpath, blocksize, roi, zarr_format):
    input_zarr = open_zarr(input_path, input_subpath)

    output_zarr = create_zarr(
        output_path,
        output_subpath,
        input_zarr.shape[-3:],
        chunks=(64,64,64),
        zarr_format=zarr_format
    )

    with create_cluster(nworkers, **worker_job_args) as segmentation_cluster:
        logger.info('Run distributed eval')
        unmerged_labels, _ = distributed_eval(
            input_zarr,
            0, # input_timeindex
            [1], # input_channels
            blocksize,
            test_output_dir,
            output_zarr,
            segmentation_cluster.client,
            blockoverlaps=(),
            mask=None,
            roi=roi,
            cellpose_model_args={
                'use_gpu': True,
                'gpu_device': '0',
                'pretrained_model': 'cpsam',
            },
            normalize_args={
                'normalize': True,
                'lowhigh': (1,99),
            },
            cellpose_eval_args={
                'do_3D': True,
                'diameter': 30,
                'min_size': 15,
                'max_size_fraction': 0.4,
                'cellprob_threshold': -8,
                'flow3D_smooth': 1,
                'batch_size': 8,
            },
            label_dist_th=1.0,        
            skip_merge_labels=True,
        )

    logger.info(f'Completed distributed segmentation: {unmerged_labels.shape}')
    return unmerged_labels

## Post-eval processing

In [7]:
import scipy
import traceback

from cellpose import utils

from dask.distributed import as_completed, Client
from segmentation.block_utils import get_block_crops, prepare_blocksize


def distributed_posteval(
    labels_zarr:zarr.Array,
    blocksize,
    dask_client: Client,
    mask=None,
    roi=None,
    labels_source_zarr:zarr.Array=None,
    labels_output_zarr:zarr.Array=None,
    min_size_voxels=-1,
    intensity_channel=-1,
    input_high_percentile=98,
):
    labels_shape = labels_zarr.shape
    blocksize = prepare_blocksize(labels_shape, blocksize)
    block_indices, block_crops = get_block_crops(
        labels_shape, blocksize, None, mask, roi,
    )
    if len(block_indices) == 0:
        return labels_zarr

    futures = dask_client.map(
        _block_posteval,
        block_indices,
        block_crops,
        labels_zarr=labels_zarr,
        source_zarr=labels_source_zarr,
        output_zarr=labels_output_zarr,
        min_size_voxels=min_size_voxels,
        intensity_channel=intensity_channel,
        input_high_percentile=input_high_percentile,
    )

    for f, r in as_completed(futures, with_results=True):
        if f.cancelled():
            tb = f.traceback()
            tb = f.traceback()
            stacktrace = ''.join(traceback.format_tb(tb))
            logger.error(f'Block error: {stacktrace}')
        else:
            bi, bc = r
            logger.info(f'Completed block {bi} at {bc}')
    
    return labels_zarr


def _block_posteval(
    block_index,
    crop,
    labels_zarr,
    min_size_voxels=-1,
    source_zarr=None,
    output_zarr=None,
    intensity_channel=-1,
    input_high_percentile=98,
):
    labels_block = labels_zarr[crop]
    logger.debug(f'Process block {block_index} at {crop}')
    if min_size_voxels > 0:
        filtered_labels_block = utils.fill_holes_and_remove_small_masks(labels_block, min_size=min_size_voxels)
    else:
        filtered_labels_block = labels_block

    mask_ids = np.unique(filtered_labels_block)
    mask_ids = mask_ids[mask_ids != 0]

    # check the input is provided and there are foreground masks
    if source_zarr is not None and intensity_channel >= 0 and mask_ids.size > 0:
        logger.debug(f'Filter block {block_index} labels based on channel {intensity_channel} intensity')
        input_coord = (0,intensity_channel) + crop
        input_block = source_zarr[input_coord]
        input_avg_label_intensity = scipy.ndimage.mean(input_block, labels=filtered_labels_block, index=mask_ids)
        input_threshold = np.percentile(input_avg_label_intensity, input_high_percentile)
        dropped_mask_ids = mask_ids[input_avg_label_intensity < input_threshold]
        filtered_labels_block[np.isin(filtered_labels_block, dropped_mask_ids)] = 0

    # write filtered labels
    if output_zarr is not None:
        output_zarr[crop] = filtered_labels_block

    return block_index, crop


def run_distributed_posteval(input_path, input_subpath, output_path, output_subpath,
                             additional_input_path, additional_input_subpath,
                             blocksize, roi, zarr_format,
                             min_size_voxels=1000,
                             intensity_channel=-1,
                             input_high_percentile=98):
   input_zarr = open_zarr(input_path, input_subpath)

   additional_input_zarr = open_zarr(additional_input_path, additional_input_subpath)

   output_zarr = create_zarr(
      output_path,
      output_subpath,
      input_zarr.shape[-3:],
      chunks=(64,64,64),
      zarr_format=zarr_format
   )

   with create_cluster(nworkers, **worker_job_args) as posteval_cluster:
      logger.info('Run post-eval step')
      posteval_labels = distributed_posteval(
         input_zarr,
         blocksize,
         posteval_cluster.client,
         mask=None,
         roi=roi,
         labels_source_zarr=additional_input_zarr,
         labels_output_zarr=output_zarr,
         min_size_voxels=min_size_voxels,
         intensity_channel=intensity_channel,
         input_high_percentile=input_high_percentile,
      )

   logger.info(f'Completed post-eval: {posteval_labels.shape}')
   return posteval_labels


In [8]:
from segmentation.distributed_cellpose import distributed_merge


def run_distributed_merge(input_path, input_subpath, output_path, output_subpath, working_dir,
                          blocksize, roi, zarr_format):
   input_zarr = open_zarr(input_path, input_subpath)

   output_zarr = create_zarr(
      output_path,
      output_subpath,
      input_zarr.shape[-3:],
      chunks=(64,64,64),
      zarr_format=zarr_format
   )

   label_dist_th = 1.0

   with create_cluster(nworkers, **worker_job_args) as merge_cluster:
      logger.info('Run merge step')
      merged_labels, _ = distributed_merge(
         input_zarr,
         blocksize,
         output_zarr,
         working_dir,
         merge_cluster.client,
         mask=None,
         roi=roi,
         label_dist_th=label_dist_th,
      )

   logger.info(f'Completed merge: {merged_labels.shape}')
   return merged_labels

## Set input/output data

In [9]:
test_data_dir = '/nrs/scicompsoft/goinac/multifish/testgudrun/fru-test'
test_input_zarrname = 'Gel1_Fru_594_dapi_1x_6x9c_VNC_stitched.zarr'

test_output_dir = f'{test_data_dir}/cg-seg'
test_labels_containername = 'labels.zarr'

test_input_container = f'{test_data_dir}/{test_input_zarrname}'
test_labels_container = f'{test_output_dir}/{test_labels_containername}'


blocksize = (256, 256, 256)
test_roi = (1260, 900, 800, 2010, 1620, 990)        # (xmin, ymin, zmin, xmax, ymax, zmax)



In [11]:
# unmerged_labels = run_distributed_eval(test_input_container, 's0', test_labels_container, 'unmerged-labels/0', blocksize, test_roi, 3)

2026-07-22 11:29:50 - zarr_io:zarr.group - DEBUG - key=s0, store_path=file:///nrs/scicompsoft/goinac/multifish/testgudrun/fru-test/Gel1_Fru_594_dapi_1x_6x9c_VNC_stitched.zarr/s0
2026-07-22 11:29:50 - MainThread:root - INFO - Create zarr container /nrs/scicompsoft/goinac/multifish/testgudrun/fru-test/cg-seg/labels.zarr:unmerged-labels/0
2026-07-22 11:29:50 - MainThread:root - INFO - Use zarrv3 sharding: {'shards': (256, 256, 256)}
2026-07-22 11:29:50 - MainThread:dask_jobqueue.lsf - DEBUG - Setting units to m from the LSF config file at ['# This file is produced automatically by lsfconfig according to\n', '# installation setup. Refer to "Administering IBM Spectrum LSF"\n', '# before changing any parameters in this file.\n', '# Any changes to the path names of LSF files must be reflected\n', '# in this file. Make these changes with caution.\n', '\n', '# After editing this file, run "lsadmin reconfig" and\n', '# "badmin mbdrestart" to apply your changes.\n', '\n', '\n', 'LSB_SHAREDIR=/mis

<Array file:///nrs/scicompsoft/goinac/multifish/testgudrun/fru-test/cg-seg/labels.zarr/unmerged-labels/0 shape=(1507, 8037, 3748) dtype=uint32>

In [12]:
merged_labels = run_distributed_merge(test_labels_container, 'unmerged-labels/0', test_labels_container, 'merged-labels/0',test_output_dir, blocksize, test_roi, 3)

2026-07-22 12:21:10 - zarr_io:zarr.group - DEBUG - key=unmerged-labels/0, store_path=file:///nrs/scicompsoft/goinac/multifish/testgudrun/fru-test/cg-seg/labels.zarr/unmerged-labels/0
2026-07-22 12:21:10 - MainThread:root - INFO - Create zarr container /nrs/scicompsoft/goinac/multifish/testgudrun/fru-test/cg-seg/labels.zarr:merged-labels/0
2026-07-22 12:21:10 - MainThread:root - INFO - Use zarrv3 sharding: {'shards': (256, 256, 256)}
2026-07-22 12:21:10 - MainThread:dask_jobqueue.lsf - DEBUG - Setting units to m from the LSF config file at ['# This file is produced automatically by lsfconfig according to\n', '# installation setup. Refer to "Administering IBM Spectrum LSF"\n', '# before changing any parameters in this file.\n', '# Any changes to the path names of LSF files must be reflected\n', '# in this file. Make these changes with caution.\n', '\n', '# After editing this file, run "lsadmin reconfig" and\n', '# "badmin mbdrestart" to apply your changes.\n', '\n', '\n', 'LSB_SHAREDIR=/

In [ ]:
logger.info('Done!')